# Author
## Steven Wu

In [ ]:
import pandas as pd
import numpy as np
import altair as alt

In [ ]:
crime = pd.read_csv("https://opendata.dc.gov/api/download/v1/items/c5a9f33ffca546babbd91de1969e742d/csv?layers=6")
# crime.to_csv("crime.csv")

# Back up data
# crime = pd.read_csv("crime.csv")

In [ ]:
crime.head()

In [ ]:
crime.columns.to_numpy()

In [ ]:
import geopandas as gpd

In [ ]:
crime_offense = crime.OFFENSE.unique().tolist()
crime_offense = ['ALL'] + crime_offense
dropdown = alt.binding_select(options = crime_offense, name = "Type of Crime: ")
selection = alt.selection_point(name = "crimeSelect", fields = ['OFFENSE'], bind = dropdown, value = "ALL")
alt.data_transformers.enable('default', max_rows=None)
dc_url = "https://opendata.dc.gov/api/download/v1/items/f6c703ebe2534fc3800609a07bad8f5b/geojson?layers=17"
dc = alt.Data(url=dc_url, format=alt.DataFormat(type='json'))
background = (
    alt.Chart(dc)
        .mark_geoshape(
            fill='lightgray',
            stroke='white'
        )
        .project(type='identity', reflectY=True)
        .properties(width=400, height=300)
)

scale_reference = (
    alt.Chart(crime)
        .mark_circle(size=0, opacity=0)
        .encode(
            longitude='LONGITUDE:Q',
            latitude='LATITUDE:Q'
        )
        .project(type='identity', reflectY=True)
)

points = (
    alt.Chart(crime)
        .mark_circle(size=10)
        .encode(
            longitude='LONGITUDE:Q',
            latitude='LATITUDE:Q',
            color=alt.Color('OFFENSE:N', title = "Offense", legend=alt.Legend(orient='right')),
            tooltip=['WARD', 'SHIFT', 'OFFENSE']
        )
        .project(type='identity', reflectY=True)
        .properties(title="Crime in Washington, DC", width = 400, height = 300)
        .add_params(selection)
        .transform_filter(
            "(crimeSelect.OFFENSE == 'ALL') | (datum.OFFENSE == crimeSelect.OFFENSE)"
        )
)

chart1 = (background + scale_reference + points).properties(
    title = "Crime in Washington, DC",
    width = 400,
    height = 400
)

chart2 = (
    alt.Chart(crime)
    .mark_bar()
    .encode(
        x=alt.X('WARD:N', title='Ward'),
        y=alt.Y('count()', title="Number of Offenses"),
        color=alt.Color('SHIFT:N', title="Shift", legend=alt.Legend(orient='bottom')),
        xOffset='SHIFT:N'
    )
    .properties(
        title=f"Offenses by Ward",
        width=400,
        height=300
    )
    .add_params(selection)
    .transform_filter(
        "(crimeSelect.OFFENSE == 'ALL') | (datum.OFFENSE == crimeSelect.OFFENSE)"
    )
    .transform_filter(
        "datum.WARD != null"
    )
)

chart = (chart1 | chart2).resolve_scale(color = "independent")

myJekyllDir = '/Users/stevenwu/Desktop/StevenWu1.github.io/assets/json/'
chart.save(myJekyllDir + 'crime_map.json')

chart

In [ ]:
crime['REPORT_DAT'] = pd.to_datetime(crime['REPORT_DAT'])
crime['MONTH'] = crime['REPORT_DAT'].dt.month

stacked_month_chart = (
    alt.Chart(crime)
    .mark_bar()
    .encode(
        x=alt.X('MONTH:N', title='Month'),
        y=alt.Y('count()', title='Number of Offenses'),
        color=alt.Color('OFFENSE:N', title='Offense Type'),
        tooltip=['MONTH:N', 'OFFENSE:N', 'count()']
    )
    .properties(
        width=600,
        height=400,
        title="Crime in DC by Month by Offense"
    )
)

myJekyllDir = '/Users/stevenwu/Desktop/StevenWu1.github.io/assets/json/'
stacked_month_chart.save(myJekyllDir + 'crime_stacked_month_chart.json')


stacked_month_chart

In [ ]:
heatmap_selection = alt.selection_point(
    fields=['WARD', 'SHIFT'],
    empty=True 
)

shift_ward = crime.groupby(['WARD', 'SHIFT']).size().reset_index(name='count')

shift_heatmap = (
    alt.Chart(shift_ward)
    .mark_rect()
    .encode(
        x=alt.X('WARD:N', title='Ward'),
        y=alt.Y('SHIFT:N', title='Shift'),
        color=alt.Color('count:Q', title='Number of Incidents', scale=alt.Scale(scheme='blues')),
        tooltip=['WARD:N', 'SHIFT:N', 'count:Q']
    )
    .add_params(heatmap_selection)
    .properties(
        width=600,
        height=300,
        title='Crime Incidents by Ward and Shift (Click a square to filter)'
    )
)

bars = (
    alt.Chart(crime)
    .mark_bar()
    .encode(
        color=alt.Color('OFFENSE:N', legend=None),
        x=alt.X('count():Q', title='Number of Incidents'),
        y=alt.Y('OFFENSE:N', title='Offense Type', sort='-x'),
        tooltip=['OFFENSE:N', 'count():Q']
    )
    .transform_filter(heatmap_selection)
    .properties(
        width=600,
        height=400,
        title='Offense Breakdown (Filtered by Heatmap Selection)'
    )
)

dashboard = alt.vconcat(shift_heatmap, bars).resolve_scale(color='independent')

myJekyllDir = '/Users/stevenwu/Desktop/StevenWu1.github.io/assets/json/'
dashboard.save(myJekyllDir + 'crime_dashboard.json')

dashboard